In [1]:
# Cell 1: Install Required Libraries
!pip install transformers datasets seqeval accelerate evaluate

In [5]:
# Cell 2: Import Libraries

import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

In [6]:
# Cell 3: Load Dataset

dataset = load_dataset("commul/universal_dependencies", "en_ewt")

print(dataset)

DatasetDict({
    dev: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 2077
    })
    train: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 12544
    })
})


In [7]:
# Cell 4: View Sample Data

sample = dataset["train"][0]

print(sample)

{'sent_id': 'weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000-0001', 'text': 'Al-Zaman : American forces killed Shaikh Abdullah al-Ani, the preacher at the mosque in the town of Qaim, near the Syrian border.', 'comments': ['newdoc id = weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000', '__SENT_ID__', 'newpar id = weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000-p0001', '__TEXT__'], 'tokens': ['Al', '-', 'Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.'], 'lemmas': ['Al', '-', 'Zaman', ':', 'American', 'force', 'kill', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.'], 'upos': [10, 1, 10, 1, 6, 0, 16, 10, 10, 10, 1, 10, 1, 8, 0, 2, 8, 0, 2, 8, 0, 2, 10, 1, 2, 8, 6, 0, 

In [8]:
# Cell 5: Create Label Names

label_names = dataset["train"].features["upos"].feature.names

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

print(label_names)
print("Number of labels:", len(label_names))

['NOUN', 'PUNCT', 'ADP', 'NUM', 'SYM', 'SCONJ', 'ADJ', 'PART', 'DET', 'CCONJ', 'PROPN', 'PRON', 'X', '_', 'ADV', 'INTJ', 'VERB', 'AUX']
Number of labels: 18


In [9]:
# Cell 6: Load Tokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [10]:
# Cell 7: Define Tokenization and Label Alignment Function

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128
    )

    labels = []

    for i, label in enumerate(examples["upos"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [12]:


tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(2000))
small_test_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(500))

print("Train samples:", len(small_train_dataset))
print("Test samples :", len(small_test_dataset))

Train samples: 2000
Test samples : 500


In [13]:
# Cell 9: Create Data Collator

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

print("Data collator ready!")

Data collator ready!


In [14]:
# Cell 10: Load DistilBERT Model

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!


In [15]:
# Cell 11: Load Seqeval Metric

seqeval = evaluate.load("seqeval")

print("Metric loaded successfully!")

Metric loaded successfully!


In [16]:
# Cell 12: Define Metrics Function

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_names[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

In [18]:
# Cell 13: Define Training Arguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

print("Training arguments ready!")

Training arguments ready!


In [19]:
# Cell 14: Create Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    # Removed 'tokenizer' parameter as it's not accepted by the Trainer constructor
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer ready!")

Trainer ready!


In [ ]:
# Cell 15: Train Model

print("Training started...")

trainer.train()

print("Training completed!")

Training started...


C:\Users\errol\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,2.144068
100,0.807896
150,0.366911
200,0.274796
250,0.206149
300,0.217157


In [ ]:
# Cell 16: Evaluate Model

results = trainer.evaluate()

print("Evaluation Results:")
print(f"Loss      : {results['eval_loss']:.4f}")
print(f"Precision : {results['eval_precision']:.4f}")
print(f"Recall    : {results['eval_recall']:.4f}")
print(f"F1 Score  : {results['eval_f1']:.4f}")
print(f"Accuracy  : {results['eval_accuracy']:.4f}")

In [ ]:
# Cell 17: Save Model

trainer.save_model("./pos_model")
tokenizer.save_pretrained("./pos_model")

print("Model saved successfully!")

In [ ]:
# Cell 18: Predict POS Tags for Custom Sentence

sentence = "John works at Google in California"

inputs = tokenizer(
    sentence.split(),
    return_tensors="pt",
    is_split_into_words=True
)

outputs = model(**inputs)

predictions = np.argmax(outputs.logits.detach().numpy(), axis=2)

predicted_labels = [id2label[p] for p in predictions[0]]

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for token, label in zip(tokens, predicted_labels):
    print(f"{token:15} --> {label}")